# Step 9 (Kaggle GPU) — Fine-tuning MACE-MPA-0 on our own PBE data for BaTiO3 and PbTiO3, then test it
1. Our DFT data (209 configurations: energy wells, strains, rattled supercells, relaxation paths; PBE/GBRV) are embedded in this notebook.
2. The DFT energies are aligned to the energy scale of the foundation model (one constant per element, least squares), because QE and VASP energies have different zeros.
3. Naive fine-tuning of MACE-MPA-0 (medium) with energies, forces and stresses.
4. Tests: errors on a held-out validation set; static benchmark (4 phases × 5 perovskites, also 3 materials **not** in the training set); NPT molecular dynamics of BaTiO3 and PbTiO3 (fine-tuned vs original model).

**Run on Kaggle:** Settings → Accelerator **GPU T4 x2**, Internet **On**. Save Version → **Save & Run All (Commit)**. Max 12 h; the notebook stops cleanly after 11 h.
Output files (Output tab): `ft_errors.csv`, `ft_static.csv`, `md_ft.csv`, `ft_train_log.txt`, `mace_ft_BTO_PTO.model`, `ft_summary.json`. Every file is written as soon as it is ready.

In [ ]:
# 1. Install MACE (2-3 min)
!pip install -q mace-torch spglib 2>&1 | tail -2
import importlib.metadata as im, torch, subprocess
print("mace-torch", im.version("mace-torch"), "| torch", torch.__version__, "| cuda", torch.cuda.is_available())
!nvidia-smi --query-gpu=name,memory.total --format=csv


In [ ]:
# 2. Our DFT training data (PBE, GBRV, Quantum ESPRESSO) - embedded
import gzip, base64, os, time, json
import numpy as np, pandas as pd
from ase.io import read, write
W = "/kaggle/working"; os.makedirs(f"{W}", exist_ok=True); os.makedirs("/tmp/ft", exist_ok=True)
T_START = time.time(); BUDGET = 11.0 * 3600
DATA = "..."
open("/tmp/ft/dft_all.extxyz", "wb").write(gzip.decompress(base64.b64decode(DATA)))
frames = read("/tmp/ft/dft_all.extxyz", ":")
for a in frames:          # make sure labels are read from the REF_* keys only
    a.info["REF_energy"] = float(a.info["REF_energy"]); a.calc = None
print(len(frames), "configurations")
print(pd.Series([a.info["config_type"] for a in frames]).value_counts().to_string())


In [ ]:
# 3. Foundation model, energy alignment, train/validation split
from mace.calculators import mace_mp, MACECalculator
calc0 = mace_mp(model="medium-mpa-0", device="cuda", default_dtype="float64")
FOUND = None
try:
    from mace.calculators.foundations_models import download_mace_mp_checkpoint
    FOUND = download_mace_mp_checkpoint("medium-mpa-0")
except Exception as e:
    print("download helper:", e)
if not FOUND or not os.path.exists(FOUND):
    import glob
    FOUND = sorted(glob.glob(os.path.expanduser("~/.cache/mace/*mpa*0*medium*")) + glob.glob(os.path.expanduser("~/.cache/mace/*mpa*")))[0]
print("foundation file:", FOUND)

# isolated-atom energies (E0) stored in the foundation model
m0 = torch.load(FOUND, map_location="cpu", weights_only=False)
zs = [int(z) for z in m0.atomic_numbers]
ae = m0.atomic_energies_fn.atomic_energies.detach().cpu().numpy().reshape(-1, len(zs))[0]
ELEM = sorted({int(z) for a in frames for z in a.numbers})
E0 = {z: float(ae[zs.index(z)]) for z in ELEM}
E0ALL = {z: float(e) for z, e in zip(zs, ae)}   # keep all 89 elements so that untrained perovskites can be tested
print("E0 (foundation):", E0)

def predict(calc, ats):
    E, F, S = [], [], []
    for a in ats:
        b = a.copy(); b.calc = calc
        E.append(b.get_potential_energy()); F.append(b.get_forces()); S.append(b.get_stress(voigt=False))
    return np.array(E), F, S

E_dft = np.array([a.info["REF_energy"] for a in frames])
E_f, F_f, S_f = predict(calc0, frames)
X = np.array([[np.sum(a.numbers == z) for z in ELEM] for a in frames], float)
delta, *_ = np.linalg.lstsq(X, E_dft - E_f, rcond=None)     # one constant per element
shift = X @ delta
for a, s in zip(frames, shift):
    a.info["REF_energy_raw"] = a.info["REF_energy"]; a.info["REF_energy"] = a.info["REF_energy"] - s
nat = np.array([len(a) for a in frames])
print("foundation vs DFT after alignment: energy MAE %.1f meV/atom" % (1000*np.mean(np.abs((E_dft - shift - E_f)/nat))))

rng = np.random.default_rng(7)
iv = set(rng.choice(len(frames), size=len(frames)//10, replace=False).tolist())
train = [a for i, a in enumerate(frames) if i not in iv]; valid = [a for i, a in enumerate(frames) if i in iv]
write("/tmp/ft/train.xyz", train, format="extxyz"); write("/tmp/ft/valid.xyz", valid, format="extxyz")
print(len(train), "train /", len(valid), "validation")


In [ ]:
# 4. Fine-tuning (about 30-60 min on a T4)
helptxt = subprocess.run(["mace_run_train", "--help"], capture_output=True, text=True).stdout
ARGS = dict(name="mace_ft_BTO_PTO", foundation_model=FOUND, multiheads_finetuning="False",
            train_file="/tmp/ft/train.xyz", valid_file="/tmp/ft/valid.xyz",
            energy_key="REF_energy", forces_key="REF_forces", stress_key="REF_stress",
            E0s=str(E0ALL), foundation_model_elements="True", loss="universal", energy_weight=10.0, forces_weight=10.0, stress_weight=100.0,
            compute_stress="True", lr=0.005, scaling="rms_forces_scaling", batch_size=4, valid_batch_size=8,
            max_num_epochs=200, patience=40, ema=None, ema_decay=0.99, amsgrad=None, default_dtype="float64",
            device="cuda", seed=1, model_dir=W, checkpoints_dir="/tmp/ft/ckpt", log_dir="/tmp/ft/logs",
            results_dir="/tmp/ft/results", save_cpu="None")
if "--foundation_model_elements" not in helptxt: ARGS["E0s"] = str(E0)
elif "FOUNDATION_MODEL_ELEMENTS" not in helptxt: ARGS["foundation_model_elements"] = None   # plain on/off flag
cmd = ["mace_run_train"]
for k, v in ARGS.items():
    if f"--{k}" not in helptxt: print("flag not available, skipped:", k); continue
    cmd.append(f"--{k}" if v is None or v == "None" else f"--{k}={v}")
print(" ".join(cmd))
t0 = time.time()
with open(f"{W}/ft_train_log.txt", "w") as log:
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        log.write(line); log.flush()
        if "Epoch" in line or "Error" in line or "error" in line or "RMSE" in line or "MAE" in line: print(line.rstrip(), flush=True)
    p.wait()
TRAIN_MIN = round((time.time()-t0)/60)
print("training finished, exit code", p.returncode, f"({TRAIN_MIN} min)")
assert p.returncode == 0, "training failed - send the end of ft_train_log.txt to Claude"
import glob
cands = [f for f in glob.glob(f"{W}/mace_ft_BTO_PTO*.model") if "compiled" not in f]
FT = sorted(cands, key=lambda f: ("stagetwo" not in f, f))[0]
if FT != f"{W}/mace_ft_BTO_PTO.model": os.replace(FT, f"{W}/mace_ft_BTO_PTO.model"); FT = f"{W}/mace_ft_BTO_PTO.model"
for f in glob.glob(f"{W}/*compiled*.model"): os.remove(f)
print("fine-tuned model:", FT)


In [ ]:
# 5. Errors on training and validation sets: original vs fine-tuned
calc_ft = MACECalculator(model_paths=FT, device="cuda", default_dtype="float64")
rows = []
for nm, calc in [("MACE-MPA-0", calc0), ("MACE-MPA-0-FT", calc_ft)]:
    for split, ats in [("train", train), ("valid", valid)]:
        E, F, S = predict(calc, ats)
        Er = np.array([a.info["REF_energy"] for a in ats]); n = np.array([len(a) for a in ats])
        dF = np.concatenate([(f - a.arrays["REF_forces"]).ravel() for f, a in zip(F, ats)])
        dS = np.concatenate([(s - np.array(a.info["REF_stress"]).reshape(3, 3)).ravel() for s, a in zip(S, ats)])
        rows.append(dict(model=nm, split=split, n=len(ats), E_MAE_meV_atom=1000*np.mean(np.abs((E - Er)/n)),
                         F_MAE_meV_A=1000*np.mean(np.abs(dF)), S_MAE_GPa=160.2177*np.mean(np.abs(dS))))
err = pd.DataFrame(rows); err.to_csv(f"{W}/ft_errors.csv", index=False); print(err.round(2).to_string(index=False))


In [ ]:
# 6. Static benchmark (same protocol as step 1): 4 phases x 5 perovskites, original vs fine-tuned
try:
    import spglib
except ImportError:
    import subprocess as sp; sp.run(['pip', 'install', '-q', 'spglib']); import spglib
def spg(at):
    try: return spglib.get_spacegroup((at.cell[:], at.get_scaled_positions(), at.numbers), symprec=1e-3)
    except Exception as e: return 'n/a'
from ase import Atoms
from ase.constraints import FixSymmetry
from ase.filters import FrechetCellFilter
from ase.optimize import BFGS
SYSTEMS = {"BaTiO3": ("Ba","Ti"), "PbTiO3": ("Pb","Ti"), "KNbO3": ("K","Nb"), "SrTiO3": ("Sr","Ti"), "KTaO3": ("K","Ta")}
DIRS = {"Pm-3m": None, "P4mm": [0,0,1], "Amm2": [1,1,0], "R3m": [1,1,1]}
def perovskite(A, B, a=4.0):
    return Atoms([A,B,"O","O","O"], scaled_positions=[[0,0,0],[.5,.5,.5],[.5,.5,0],[.5,0,.5],[0,.5,.5]], cell=np.eye(3)*a, pbc=True)
def distort(at, d, dB=0.10, dO=-0.06):
    at = at.copy(); d = np.array(d, float); d /= np.linalg.norm(d); p = at.get_positions(); p[1] += dB*d; p[2:] += dO*d; at.set_positions(p); return at
def relax(at, calc, fmax=5e-4, steps=4000):
    at = at.copy(); at.calc = calc; at.set_constraint(FixSymmetry(at, symprec=1e-4))
    opt = BFGS(FrechetCellFilter(at), logfile=None); conv = opt.run(fmax=fmax, steps=steps); at.set_constraint(); return at, bool(conv)
srows = []
for nm, calc in [("MACE-MPA-0", calc0), ("MACE-MPA-0-FT", calc_ft)]:
    for mat, (A, B) in SYSTEMS.items():
        for ph, d in DIRS.items():
            try:
                at, conv = relax(perovskite(A, B) if d is None else distort(perovskite(A, B), d), calc)
            except Exception as e:
                print("skipped", nm, mat, ph, ":", e); continue
            cp = at.cell.cellpar()
            srows.append(dict(model=nm, material=mat, phase=ph, in_training=mat in ("BaTiO3", "PbTiO3"), E_eV=at.get_potential_energy(),
                              a=cp[0], b=cp[1], c=cp[2], alpha=cp[3], volume=at.get_volume(), converged=conv,
                              spg=spg(at)))
st = pd.DataFrame(srows)
st["dE_meV_per_fu"] = 1000*(st.E_eV - st.groupby(["model", "material"]).E_eV.transform(lambda s: s.iloc[0]))
st["c_over_a"] = st.c/st.a
st.to_csv(f"{W}/ft_static.csv", index=False)
print(st.pivot_table(index=["material", "phase"], columns="model", values="dE_meV_per_fu").round(1).to_string())
print(st[st.phase == "P4mm"][["model", "material", "c_over_a"]].round(4).to_string(index=False))
json.dump(dict(foundation=os.path.basename(FOUND), n_train=len(train), n_valid=len(valid), args={k: str(v) for k, v in ARGS.items() if k != "E0s"},
               E0=E0, delta_per_element=dict(zip(map(int, ELEM), map(float, delta))), mace_version=im.version("mace-torch"),
               train_minutes=TRAIN_MIN), open(f"{W}/ft_summary.json", "w"), indent=1)


In [ ]:
# 7. NPT molecular dynamics (same protocol as step 7): fine-tuned vs original, BaTiO3 then PbTiO3
from ase import units
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from ase.md.nptberendsen import Inhomogeneous_NPTBerendsen
from ase.neighborlist import neighbor_list
del calc_ft, calc0; torch.cuda.empty_cache()
MDCALC = {"MACE-MPA-0-FT": lambda: MACECalculator(model_paths=FT, device="cuda", default_dtype="float32"),
          "MACE-MPA-0": lambda: mace_mp(model="medium-mpa-0", device="cuda", default_dtype="float32")}
SETUP = {"BaTiO3": dict(A="Ba", B="Ti", gs=[1, 1, 1], T=list(range(50, 901, 50))),
         "PbTiO3": dict(A="Pb", B="Ti", gs=[0, 0, 1], T=list(range(200, 1501, 100)))}
N, DT, NSTEP, NAVG = 4, 2.0, 2500, 1250
OUT = f"{W}/md_ft.csv"
def unit(A, B, d, a=4.0):
    at = perovskite(A, B, a); d = np.array(d, float); d /= np.linalg.norm(d); p = at.get_positions(); p[1] += 0.1*d; p[2:] -= 0.06*d; at.set_positions(p); return at
def cages(at, B):
    i, j = neighbor_list("ij", at, 2.2); sym = np.array(at.get_chemical_symbols())
    idx = [k for k in range(len(at)) if sym[k] == B]
    cage = {b: [jj for ii, jj in zip(i, j) if ii == b and sym[jj] == "O"] for b in idx}
    assert all(len(v) == 6 for v in cage.values()); return idx, cage
def order_param(at, idx, cage):
    pos = at.get_positions(); cell = at.cell[:]; d = []
    for b in idx:
        v = pos[cage[b]] - pos[b]; v = v - np.round(v @ np.linalg.inv(cell)) @ cell; d.append(-v.mean(0))
    d = np.array(d); return d.mean(0), np.linalg.norm(d, axis=1).mean()
done = pd.DataFrame()
stop = False
for mat in ["BaTiO3", "PbTiO3"]:
    s = SETUP[mat]
    for nm in ["MACE-MPA-0-FT", "MACE-MPA-0"]:
        if stop: break
        calc = MDCALC[nm]()
        uc, _ = relax(unit(s["A"], s["B"], s["gs"]), calc, fmax=1e-3, steps=2000)
        at = uc.repeat((N, N, N)); at.calc = calc
        idx, cage = cages(perovskite(s["A"], s["B"]).repeat((N, N, N)), s["B"])
        MaxwellBoltzmannDistribution(at, temperature_K=s["T"][0]); Stationary(at)
        for T in s["T"]:
            if time.time() - T_START > BUDGET: print("time budget reached, stopping"); stop = True; break
            t1 = time.time()
            dyn = Inhomogeneous_NPTBerendsen(at, timestep=DT*units.fs, temperature_K=T, pressure_au=0.0,
                                             taut=100*units.fs, taup=1000*units.fs, compressibility_au=4.57e-5/units.bar)
            rec = []
            def sample():
                if dyn.nsteps >= NSTEP - NAVG and dyn.nsteps % 10 == 0:
                    P, mag = order_param(at, idx, cage); cp = at.cell.cellpar()
                    rec.append([*P, np.linalg.norm(P), mag, cp[0]/N, cp[1]/N, cp[2]/N, at.get_temperature()])
            dyn.attach(sample, interval=1); dyn.run(NSTEP)
            r = np.array(rec).mean(0); abc = sorted(r[5:8])
            row = dict(model=nm, material=mat, T_target=T, T_md=r[8], Px=r[0], Py=r[1], Pz=r[2], P_abs=r[3], local_disp=r[4],
                       a=r[5], b=r[6], c=r[7], tetragonality=abc[2]/abc[0], time_s=round(time.time()-t1))
            done = pd.concat([done, pd.DataFrame([row])], ignore_index=True); done.to_csv(OUT, index=False)
            print(f"{nm:14s} {mat} T={T:5d} K  |<d>|={r[3]:.3f} A  <|d|>={r[4]:.3f} A  c/a~{abc[2]/abc[0]:.4f}  {(time.time()-t1)/NSTEP:.3f} s/step", flush=True)
        del calc; torch.cuda.empty_cache()
print("done ->", OUT, f"| total {(time.time()-T_START)/3600:.1f} h")
